# Does the new LTF schedule change the spectrum, or only which bins are reported?

`new_ltf_plan` places resolution and averages differently across the band. The estimator behind it stays the same: the same window, the same per-segment detrend, and the same one-sided ASD. The ASD is the square root of the one-sided power spectral density. Only the plan changes. Its docstring describes three regimes of a single loop. At low frequency it uses the ordinary LTF compromise between resolution and the number of averages. Through the middle of the band the segment length decreases exponentially toward `Lmin`, arranged so the plan comes out near `Jdes` frequencies in total. At high frequency the segment length stays at `Lmin`, and the frequencies advance in equal steps of $f_s/L_{\min}$. That last stretch is a linear ramp.

Both plans estimate the same quantity, so the two ASDs of one record should agree in level. They report different frequencies, and each bin is averaged a different number of times, so the scatter can differ too. A step where two regimes meet would be a stitching error in the plan. A vertical offset that persists over a stretch of the band would be the same kind of failure.

The record is a GRACE Follow-On ranging series at $f_s = 9.664\,\mathrm{Hz}$. `compute_spectrum` is called twice, with a Kaiser window, 75% overlap, `Jdes` 1000, and `force_target_nf`. Each call adjusts `Jdes` until its plan has 1000 bins. One call uses `vectorized_ltf`, the analyzer default. The other uses `new_ltf`. The Kaiser window is shaped for the default peak sidelobe level of 200 dB. `order=0` removes the mean of each segment. `Lmin` and `bmin` stay at their defaults of 1. The plotted quantity is the one-sided ASD.


In [ ]:
import os
import requests
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

from speckit import compute_spectrum
from speckit.plotting import default_rc

plt.rcParams.update(default_rc)

In [ ]:
# GRACE Follow-On LRI phase, calendar week 52 of 2022.
# The Drive share link is a preview page. This URL is the file itself;
# confirm=t accepts the virus-scan warning Google returns for it.
# Contact the author at mikedovale@pm.me if the link is broken
file_id = "15IdylJxn79FEp-8If3rYTYOHRRknMKA-"
url = (
    "https://drive.usercontent.google.com/download"
    f"?id={file_id}&export=download&confirm=t"
)

filename = "data.mat"

# Skip the download only when the existing file has a MATLAB header.
# The windows notebook reads this same file.
needs_download = True
if os.path.exists(filename):
    with open(filename, "rb") as f:
        needs_download = f.read(6) != b"MATLAB"

if needs_download:
    response = requests.get(url)
    response.raise_for_status()
    if not response.content.startswith(b"MATLAB"):
        raise RuntimeError(
            "Google Drive did not return a MAT-file "
            f"(content-type {response.headers.get('content-type')})"
        )
    with open(filename, "wb") as f:
        f.write(response.content)
    print("File downloaded.")
else:
    print("File already exists. Skipping download.")

# Load the .mat file into a NumPy array
data = loadmat(filename)
x = np.array(data["RawPhase"][:, 0])
print(x)

fs = 9.664

In [ ]:
xlabel = "Sample"
ylabel = "Signal"
title = "GRACE Follow On LRI range in meters - Calendar week 52, 2022"

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x, label="test", color="k")
ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)
ax.set_title(title)
fig.tight_layout()
fig.align_ylabels()
plt.show()

In [ ]:
olap = 0.75
Jdes = 1000
force_target_nf = True
res = {}

res["ltf"] = compute_spectrum(
    x,
    fs=fs,
    win="kaiser",
    olap=olap,
    Jdes=Jdes,
    force_target_nf=force_target_nf,
    scheduler="vectorized_ltf",
)

In [ ]:
res["new_ltf"] = compute_spectrum(
    x,
    fs=fs,
    win="kaiser",
    olap=olap,
    Jdes=Jdes,
    force_target_nf=force_target_nf,
    scheduler="new_ltf",
)

In [ ]:
fig, ax = plt.subplots()

res["ltf"].plot(ax=ax, label="LTF")
res["new_ltf"].plot(ax=ax, label="New LTF", ls='--')

ax.legend()
plt.show()

## Reading the result

The time series is the record. The spectrum figure overlays the `vectorized_ltf` estimate, drawn solid, and the `new_ltf` estimate, drawn dashed.

At low frequency both plans use the same compromise, so the two traces should nearly overlie. Above the transition the new plan sets the segment length from its exponential schedule and then holds it at `Lmin`. The dashed trace can sit on different frequencies, and it can look smoother or noisier. Its level should stay with the solid trace. A step where the regimes meet, or an offset that persists over a band, comes from the scheduler. The record is the same for both curves.

`Lmin` $= 1$ lets the middle regime drive the segment length down to a few samples. The resolution there is the width of the Kaiser main lobe, several bins. The spacing between reported frequencies is a smaller number. The high-frequency part of the dashed curve is averaged over that lobe width.
